# Enfoques de Integración en Profundidad

**Objetivo:** Detallar cada estrategia (Big-Bang, Top-Down, Bottom-Up, Sandwich), sus ventajas, desventajas, y cómo implementarlas con stubs y drivers en el gestor de tareas.

---

## 1. Big-Bang

**Idea:** Integrar todos los módulos de una vez y probar el sistema completo sin sustitutos.

**Ventajas:**
- Sencillo si el sistema es pequeño.
- No requiere escribir stubs ni drivers.

**Desventajas:**
- Si algo falla, es muy difícil localizar el origen del error.
- No se puede probar nada hasta que **todos** los módulos están listos.
- Los fallos de integración aparecen todos al mismo tiempo.

**¿Cuándo usarlo?** Solo en sistemas muy pequeños o prototipos desechables.

En nuestro gestor de tareas, las pruebas iniciales en `tests/test_service_integration.py` son esencialmente big-bang: usan todos los módulos reales sin ningún sustituto. Como verás, pasan aunque el sistema tiene bugs graves.

---

## 2. Top-Down

**Idea:** Empezar probando el módulo de más alto nivel (`Service`) y reemplazar los módulos inferiores por **stubs**. A medida que los módulos inferiores se completan, se sustituyen los stubs por módulos reales.

**Ventajas:**
- Se puede probar la lógica principal desde el principio.
- Los defectos críticos de flujo se detectan temprano.

**Desventajas:**
- Los módulos inferiores se prueban tarde.
- Los stubs pueden volverse complejos si la dependencia tiene lógica elaborada.

**Diagrama:**
```
FASE 1:  Tests → TaskService → [StubStorage]  + [StubNotifier]
FASE 2:  Tests → TaskService → [Storage real] + [StubNotifier]
FASE 3:  Tests → TaskService → [Storage real] + [Notifier real]
```

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from src.service import TaskService

class StubStorage:
    def __init__(self, initial_tasks=None, fail_on_save=False):
        self._tasks = initial_tasks or []
        self._fail_on_save = fail_on_save
    def load(self):
        return list(self._tasks)
    def save(self, tasks):
        if self._fail_on_save:
            raise IOError("Fallo simulado de escritura")
        self._tasks = list(tasks)

class StubNotifier:
    def __init__(self, fail=False):
        self.messages = []
        self._fail = fail
    def send(self, message):
        if self._fail:
            raise ConnectionError("Fallo simulado de red")
        self.messages.append(message)

print("Stubs disponibles.")

In [ ]:
# ══════════════════════════════════════════════════════
# PRUEBAS TOP-DOWN (Fase 1) — Service con stubs
# ══════════════════════════════════════════════════════

def test_td_add_task_nueva():
    service = TaskService(StubStorage(), StubNotifier())
    assert service.add_task("Nueva tarea") is True
    print("✓ [TD] add_task → True para tarea nueva")

def test_td_add_task_duplicada():
    existing = [{"title": "Existente", "done": False}]
    service = TaskService(StubStorage(initial_tasks=existing), StubNotifier())
    assert service.add_task("Existente") is False
    print("✓ [TD] add_task → False para tarea duplicada")

def test_td_add_task_guarda_en_storage():
    storage = StubStorage()
    service = TaskService(storage, StubNotifier())
    service.add_task("Persistir")
    assert len(storage._tasks) == 1
    assert storage._tasks[0]["title"] == "Persistir"
    assert storage._tasks[0]["done"] is False
    print("✓ [TD] add_task persiste la tarea correctamente en storage")

def test_td_notifier_falla_propaga_excepcion():
    """Detecta el bug: tarea guardada pero excepción propagada sin rollback."""
    storage = StubStorage()
    service = TaskService(storage, StubNotifier(fail=True))
    try:
        service.add_task("Con notifier fallido")
    except ConnectionError:
        guardada = len(storage._tasks) > 0
        print(f"✓ [TD] ConnectionError propagado. ¿Tarea guardada? {guardada}")
        print("  ⚠ Bug de integración: estado inconsistente (tarea guardada sin notificación).")

def test_td_complete_task_actualiza_estado():
    storage = StubStorage(initial_tasks=[{"title": "Completar", "done": False}])
    service = TaskService(storage, StubNotifier())
    assert service.complete_task("Completar") is True
    assert storage._tasks[0]["done"] is True
    print("✓ [TD] complete_task actualiza el estado de la tarea")

def test_td_notifier_recibio_mensaje_correcto():
    notifier = StubNotifier()
    service = TaskService(StubStorage(), notifier)
    service.add_task("Verificar mensaje")
    assert len(notifier.messages) == 1
    assert "Verificar mensaje" in notifier.messages[0]
    print("✓ [TD] notifier.send() recibió un mensaje con el título de la tarea")

print("═" * 50)
print("PRUEBAS TOP-DOWN")
print("═" * 50)
test_td_add_task_nueva()
test_td_add_task_duplicada()
test_td_add_task_guarda_en_storage()
test_td_notifier_falla_propaga_excepcion()
test_td_complete_task_actualiza_estado()
test_td_notifier_recibio_mensaje_correcto()

---

## 3. Bottom-Up

**Idea:** Empezar probando los módulos de más bajo nivel (`Storage`) directamente con un **driver**. Progresivamente subimos hacia la lógica de negocio.

**Ventajas:**
- Los módulos de infraestructura quedan muy probados.
- Los errores en capas bajas se detectan temprano.

**Desventajas:**
- La lógica de alto nivel se prueba tarde.
- Los drivers pueden requerir esfuerzo considerable.

**Diagrama:**
```
FASE 1:  [Driver] → TaskStorage
FASE 2:  [Driver] → TaskStorage + Notifier
FASE 3:  TaskService → TaskStorage + Notifier
```

In [ ]:
import tempfile, json
from src.storage import TaskStorage

# ══════════════════════════════════════════════════════
# DRIVER para TaskStorage — Bottom-Up
# ══════════════════════════════════════════════════════

def bu_setup():
    """Crea un archivo temporal limpio para cada caso."""
    f = tempfile.NamedTemporaryFile(suffix='.json', delete=False)
    f.close()
    os.unlink(f.name)
    return TaskStorage(f.name), f.name

def test_bu_inicia_vacio():
    s, path = bu_setup()
    try:
        assert s.load() == []
        print("✓ [BU] Storage inicia vacío")
    finally:
        os.unlink(path)

def test_bu_save_load_roundtrip():
    s, path = bu_setup()
    try:
        tasks = [{"title": "T1", "done": False}]
        s.save(tasks)
        assert s.load() == tasks
        print("✓ [BU] save/load roundtrip preserva los datos")
    finally:
        os.unlink(path)

def test_bu_save_sobrescribe():
    s, path = bu_setup()
    try:
        s.save([{"title": "Vieja", "done": False}])
        s.save([{"title": "Nueva", "done": True}])
        loaded = s.load()
        assert len(loaded) == 1 and loaded[0]["title"] == "Nueva"
        print("✓ [BU] save sobrescribe la lista completa")
    finally:
        os.unlink(path)

def test_bu_persistencia_entre_instancias():
    s, path = bu_setup()
    try:
        s.save([{"title": "Persistente", "done": False}])
        s2 = TaskStorage(path)  # nueva instancia, mismo archivo
        assert s2.load() == [{"title": "Persistente", "done": False}]
        print("✓ [BU] Datos persisten entre instancias diferentes")
    finally:
        os.unlink(path)

def test_bu_formato_json_en_disco():
    s, path = bu_setup()
    try:
        s.save([{"title": "JSON", "done": False}])
        with open(path) as f:
            data = json.load(f)
        assert isinstance(data, list) and data[0]["title"] == "JSON"
        print("✓ [BU] Formato en disco es JSON válido y legible")
    finally:
        os.unlink(path)

print("═" * 50)
print("PRUEBAS BOTTOM-UP — Driver de TaskStorage")
print("═" * 50)
test_bu_inicia_vacio()
test_bu_save_load_roundtrip()
test_bu_save_sobrescribe()
test_bu_persistencia_entre_instancias()
test_bu_formato_json_en_disco()

---

## 4. Sandwich (Híbrido)

**Idea:** Combinar Top-Down y Bottom-Up. Se prueba la capa superior con stubs para la inferior, y la inferior con drivers, hasta que ambas están listas para integrarse realmente.

**Ventajas:**
- Buen equilibrio entre detección temprana y cobertura real.
- Útil cuando distintos equipos trabajan las capas en paralelo.

En nuestro caso, usamos **`Storage` real** (ya probado con el driver) y un **`StubNotifier`** (la red es externa e impredecible):

```
SANDWICH:  Tests → TaskService (real) → Storage (real) + StubNotifier
```

In [ ]:
from src.service import TaskService
from src.storage import TaskStorage

def make_sandwich():
    """Crea el servicio con storage real y notifier stub."""
    f = tempfile.NamedTemporaryFile(suffix='.json', delete=False)
    f.close()
    os.unlink(f.name)
    storage = TaskStorage(f.name)
    service = TaskService(storage, StubNotifier())
    return service, storage, f.name

def test_sw_add_y_persiste_en_disco():
    """[SW] Verifica la integración real Service → Storage (archivo)."""
    service, storage, path = make_sandwich()
    try:
        service.add_task("Comprar pan")
        # Verificación directa en el archivo — integración real
        with open(path) as f:
            data = json.load(f)
        assert data == [{"title": "Comprar pan", "done": False}]
        print("✓ [SW] add_task persiste correctamente en el archivo JSON")
    finally:
        os.unlink(path)

def test_sw_complete_modifica_archivo():
    service, storage, path = make_sandwich()
    try:
        service.add_task("Completar")
        service.complete_task("Completar")
        with open(path) as f:
            data = json.load(f)
        assert data[0]["done"] is True
        print("✓ [SW] complete_task actualiza 'done' en el archivo")
    finally:
        os.unlink(path)

def test_sw_orden_de_insercion():
    service, storage, path = make_sandwich()
    try:
        for title in ["Primera", "Segunda", "Tercera"]:
            service.add_task(title)
        tasks = service.list_tasks()
        assert [t["title"] for t in tasks] == ["Primera", "Segunda", "Tercera"]
        print("✓ [SW] Orden de inserción se preserva en el archivo")
    finally:
        os.unlink(path)

print("═" * 50)
print("PRUEBAS SANDWICH (Storage real + StubNotifier)")
print("═" * 50)
test_sw_add_y_persiste_en_disco()
test_sw_complete_modifica_archivo()
test_sw_orden_de_insercion()

---

## 5. Comparación de los cuatro enfoques

| Aspecto | Big-Bang | Top-Down | Bottom-Up | Sandwich |
|---------|----------|----------|-----------|----------|
| **Velocidad de las pruebas** | Lento (I/O real) | Muy rápido | Más lento | Medio |
| **Detección de bugs en capa baja** | Tarde | Tarde | Temprana | Temprana |
| **Detección de bugs de flujo** | Tardía | Temprana | Tardía | Temprana |
| **Requiere infraestructura real** | Sí | No | Sí | Parcial |
| **Facilidad para el diagnóstico** | Muy difícil | Fácil | Fácil | Fácil |

---

## 6. Ejercicios para entregar

1. **Top-Down:** Escribe una prueba que verifique que `complete_task()` sobre una tarea inexistente retorna `False` y no llama a `storage.save()`.

2. **Bottom-Up:** Amplía el driver con un caso que intente cargar un archivo JSON corrupto. ¿Qué excepción lanza? ¿Debería el módulo manejarla?

3. **Sandwich:** Agrega un test que verifique el comportamiento cuando el notifier falla: ¿la tarea queda guardada o no? Documenta el bug encontrado y propone cómo corregirlo.

4. **Reflexión:** ¿Por qué en el enfoque Sandwich usamos el storage real pero no el notifier real? ¿Qué criterios guían esta decisión en un proyecto real?